# WLAN-Kommunikation mit ESP32

Dieses Notebook implementiert die drahtlose Steuerdatenübertragung vom Laptop zu den Fahrzeugen über UDP.

## Funktionen
- Gamepad-Abfrage
- Datenpaket-Erstellung
- Zyklische Übertragung an ESP32

## Hinweis
IP-Adresse und Port müssen an das lokale Netzwerk angepasst werden.


In [ ]:
import socket
import struct
import time
import pygame

MAGIC = 0xA7
VERSION = 1
SEND_HZ = 100

ESP_IP   = "XXX" # ESP-IP hier einsetzen
ESP_PORT = 4210
TANK_ID  = 1
JOY_IDX  = 0

# Packet: magic, version, tank_id, seq, axisY, axisX, axisRX
PKT_FMT = "!BBBBhhh"

AX_X  = 0
AX_Y  = 1
AX_RX = 2

DEADZONE = 0.12

INVERT_Y  = False
INVERT_X  = False
INVERT_RX = False

def clamp(x, lo, hi):
    return lo if x < lo else hi if x > hi else x

def apply_deadzone(v, dz):
    return 0.0 if abs(v) < dz else v

def to_i16_1000(v):
    return int(round(clamp(v, -1.0, 1.0) * 1000))

pygame.init()
pygame.joystick.init()

if pygame.joystick.get_count() == 0:
    raise RuntimeError("Kein Gamepad erkannt.")

js = pygame.joystick.Joystick(JOY_IDX)
js.init()

print("Gamepad:", js.get_name())
print("Achsen:", js.get_numaxes())

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
seq = 0

try:
    while True:
        pygame.event.pump()

        x  = js.get_axis(AX_X)  if js.get_numaxes() > AX_X  else 0.0
        y  = js.get_axis(AX_Y)  if js.get_numaxes() > AX_Y  else 0.0
        rx = js.get_axis(AX_RX) if js.get_numaxes() > AX_RX else 0.0

        if INVERT_X:
            x = -x
        if INVERT_Y:
            y = -y
        if INVERT_RX:
            rx = -rx

        x  = apply_deadzone(x, DEADZONE)
        y  = apply_deadzone(y, DEADZONE)
        rx = apply_deadzone(rx, DEADZONE)

        axisX  = to_i16_1000(x)
        axisY  = to_i16_1000(y)
        axisRX = to_i16_1000(rx)

        pkt = struct.pack(
            PKT_FMT,
            MAGIC,
            VERSION,
            TANK_ID & 0xFF,
            seq & 0xFF,
            axisY,
            axisX,
            axisRX
        )

        sock.sendto(pkt, (ESP_IP, ESP_PORT))

        print(f"axisX={axisX:5d}  axisY={axisY:5d}  axisRX={axisRX:5d}", end="\r")

        seq = (seq + 1) & 0xFF
        time.sleep(1.0 / SEND_HZ)

except KeyboardInterrupt:
    print("\nGestoppt.")

finally:
    sock.close()
    pygame.quit()